In [ ]:
#source activate newEnv
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
library(sctransform)
library(reticulate)
library('Biobase')
library(gplots)
library(BiocParallel)
library(cowplot)
library(GenomeInfoDb)

In [ ]:
atac_obj  <- readRDS(file.path(DATA_DIR, 'atac_obj.rds'))
npod_rna  <- readRDS(file.path(DATA_DIR, 'rna_obj.rds'))

gene.activities <- GeneActivity(atac_obj)
atac_obj[["ACTIVITY_allFeatures"]] <- CreateAssayObject(counts = gene.activities)

DefaultAssay(atac_obj) <- "ACTIVITY_allFeatures"
atac_obj <- NormalizeData(atac_obj)
atac_obj <- ScaleData(atac_obj, features = rownames(atac_obj))

# snATAC_05 — Label transfer from snRNA to snATAC

## Execution order — IMPORTANT

The original `data_processing.md` documents the notebook order as 03 → 04 → 05.
**This is incorrect.** The actual required execution order is:

1. `snATAC_03` — merge samples and initial clustering
2. **`snATAC_05` (this notebook, cells 1–11)** — label transfer; produces `snATAC_Lt_filt05AcinarSum.rds`
3. `snATAC_04` — manual doublet removal using promoter accessibility; reads the output above
4. **`snATAC_05` (this notebook, final acinar-sum filter cells)** — re-run after snATAC_04 to apply the cleaned barcode list

This cross-dependency was discovered during reproduction (repro branch) by tracing file
inputs/outputs.  It contradicts the authors' documented order but matches the actual
file names produced and consumed.

---

## Path configuration (repro branch)

Hardcoded `~/data/` paths replaced with `Sys.getenv()` calls (see setup cell below).

| Variable | Default | Purpose |
|---|---|---|
| `NPOD_DATA_DIR` | `/data` | Root data directory |
| `NPOD_LT_DIR` | `$NPOD_DATA_DIR/snATAC_LT_files` | Label-transfer intermediates and final outputs |

In [ ]:
rm(list = ls())
DATA_DIR <- Sys.getenv('NPOD_DATA_DIR', '/data')
LT_DIR   <- Sys.getenv('NPOD_LT_DIR',  file.path(DATA_DIR, 'snATAC_LT_files'))

npod_rna  <- readRDS(file.path(DATA_DIR, 'rna_obj.rds'))
atac_curr <- readRDS(file.path(LT_DIR,   'atac_obj_subset1.rds'))

Sys.time()
transfer.anchors <- FindTransferAnchors(reference = npod_rna, query = atac_curr,
    features = VariableFeatures(object = npod_rna),
    reference.assay = "RNA", query.assay = "ACTIVITY_allFeatures", reduction = "cca")

celltype.predictions <- TransferData(anchorset = transfer.anchors,
    refdata = npod_rna$subtypes,
    weight.reduction = atac_curr[["lsi"]], dims = 2:30)

atac_curr <- AddMetaData(atac_curr, metadata = celltype.predictions)
options(repr.plot.width=20, repr.plot.height=10)
p1 <- DimPlot(atac_curr, group.by = "predicted.id", label = TRUE) + NoLegend()
p1

saveRDS(atac_curr, file = file.path(LT_DIR, 'atac_obj_subset1_LabelTransfer.rds'))
Sys.time()

In [ ]:
atac1 <- readRDS(file.path(LT_DIR, 'atac_obj_subset1_LabelTransfer.rds'))
atac2 <- readRDS(file.path(LT_DIR, 'atac_obj_subset2_LabelTransfer.rds'))
atac3 <- readRDS(file.path(LT_DIR, 'atac_obj_subset3_LabelTransfer.rds'))

adata_mergeLT <- merge(atac1, y = c(atac2, atac3))

In [ ]:
#### example of how to run one of the smaller objects
rm(list = ls())
npod_rna <- readRDS(file= '~/data/rna_obj.rds')
atac_curr <- readRDS(file='~/data/atac_obj_subset1.rds')


Sys.time()
# Identify anchors
transfer.anchors <- FindTransferAnchors(reference = npod_rna, query = atac_curr, features = VariableFeatures(object = npod_rna),
    reference.assay = "RNA", query.assay = "ACTIVITY_allFeatures", reduction = "cca")

celltype.predictions <- TransferData(anchorset = transfer.anchors, refdata = npod_rna$subtypes,
    weight.reduction = atac_curr[["lsi"]], dims = 2:30)

atac_curr <- AddMetaData(atac_curr, metadata = celltype.predictions)#,col.name= '2ndLT_predictedID')
options(repr.plot.width=20, repr.plot.height=10)

p1 <- DimPlot(atac_curr, group.by = "predicted.id", label = TRUE) + NoLegend() 
p1

saveRDS(atac_curr, file = '~/data/snATAC_LT_files/atac_obj_subset1_LabelTransfer.rds')
Sys.time()


In [ ]:
#### then merge of objects to get a whole label transferred object
atac1 <- readRDS('~/data/snATAC_LT_files/atac_obj_subset1_LabelTransfer.rds')
atac2 <- readRDS('~/data/snATAC_LT_files/atac_obj_subset2_LabelTransfer.rds')
atac3 <- readRDS('~/data/snATAC_LT_files/atac_obj_subset3_LabelTransfer.rds')

adata_mergeLT <- merge(atac1, y = c(atac2,atac3))


In [ ]:
## recluster
DefaultAssay(adata_mergeLT) <- 'ATAC'
adata_mergeLT <- RunTFIDF(adata_mergeLT)
adata_mergeLT <- FindTopFeatures(adata_mergeLT, min.cutoff='q0', verbose=FALSE)
adata_mergeLT <- RunSVD(adata_mergeLT)

hm_atac <- HarmonyMatrix(Embeddings(adata_mergeLT, reduction='lsi'),adata_mergeLT@meta.data,  c("library","sex","technology"), do_pca=FALSE,plot_convergence = TRUE, verbose = TRUE)
adata_mergeLT[['harmony.atac']] <- CreateDimReducObject(embeddings=hm_atac, key='atac_', assay='ATAC')
adata_mergeLT <- RunUMAP(adata_mergeLT, dims=2:30, reduction='harmony.atac', reduction.name='umap.atac', reduction.key='atacUMAP_')

adata_mergeLT <- FindNeighbors(object = adata_mergeLT, reduction = 'harmony.atac', dims = 2:30)
adata_mergeLT <- FindClusters(object = adata_mergeLT, algorithm=4,resolution = 0.5,method = "igraph") 

DimPlot(object = adata_mergeLT, label = TRUE) + NoLegend()
DimPlot(object = adata_mergeLT,group.by = "predicted.id", label = TRUE) + NoLegend()


In [ ]:
adata_mergeLT$AcinarSumPrediction <- rowSums(adata_mergeLT@meta.data[,c("prediction.score.Acinar_1", "prediction.score.Acinar_2",
                                                                       "prediction.score.Acinar_3","prediction.score.Acinar_4",
                                                                       "prediction.score.Acinar_5","prediction.score.Acinar_6")])
acinar <- c("Acinar_1_2_6","Acinar_3","Acinar_4","Acinar_5")

dir.create(LT_DIR, showWarnings=FALSE, recursive=TRUE)
to_remove <- file.path(LT_DIR, "filter05_acinarSum.txt")
write.table(adata_mergeLT[[]][adata_mergeLT$FinalSubtypes %in% acinar & adata_mergeLT$AcinarSumPrediction < 0.5, c()],
            to_remove, col.names=FALSE, quote=FALSE)

In [ ]:
to_removeFinal <- file.path(LT_DIR, "filter05_acinarSum.txt")
remove.cells1 <- trimws(scan(to_removeFinal, what='', sep='\n'), which='right', whitespace=' ')

remove <- c(remove.cells1)
print(length(remove))
adata_mergeLT$remove_cells <- (Cells(adata_mergeLT) %in% remove)
adata_mergeLT_filt05 <- subset(adata_mergeLT, subset=remove_cells==FALSE)
adata_mergeLT_filt05

In [ ]:
#### since acinar subtypes did not recapitulate well in chromatin assay, we derived a cumulative acinar score 
#### for all predicted acinar cells, then retained acinar cells with a cumulative score > 0.5

adata_mergeLT$AcinarSumPrediction = rowSums(adata_mergeLT@meta.data[,c("prediction.score.Acinar_1", "prediction.score.Acinar_2",
                                                                       "prediction.score.Acinar_3","prediction.score.Acinar_4",
                                                                      "prediction.score.Acinar_5","prediction.score.Acinar_6")])
acinar <- c("Acinar_1_2_6","Acinar_3","Acinar_4","Acinar_5")

outdir2 <-'~/data/snATAC_LT_files/'
to_remove <- file.path(outdir2,"filter05_acinarSum.txt")
write.table(adata_mergeLT[[]][adata_mergeLT$FinalSubtypes %in% acinar & adata_mergeLT$AcinarSumPrediction < 0.5, c()], 
            to_remove, col.names=FALSE, quote=FALSE)

In [ ]:
saveRDS(adata_mergeLT_filt05, file.path(LT_DIR, 'snATAC_Lt_filt05AcinarSum.rds'))

In [ ]:
options(repr.plot.width=8, repr.plot.height=7)
DimPlot(object = adata_mergeLT_filt075,group.by = "FinalSubtypes",raster=FALSE, label = TRUE) + NoLegend()

In [ ]:
saveRDS(adata_mergeLT_filt05,'~/data/snATAC_LT_files/snATAC_Lt_filt05AcinarSum.rds')